# Step 6: Model Comparison — WAIC and LOO

We compare m1 (no pooling) and m2 (hierarchical) using information criteria.

- **WAIC** (Widely Applicable Information Criterion): estimates out-of-sample predictive accuracy, computed from the log-likelihood of each observation
- **LOO** (Leave-One-Out cross-validation): gold standard, holds out each observation and asks how well the model predicts it

Both penalise complexity: a model with more effective parameters gets penalised even if it fits better in-sample.

**Key question**: does the partial pooling in m2 actually improve out-of-sample prediction, or is the added structure just overfitting?

In [ ]:
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 100
plt.style.use('seaborn-v0_8-whitegrid')

# Load both fitted models (re-run 04_fit_models.ipynb if files not found)
idata_m1 = az.from_netcdf('../../data/idata_m1.nc')
idata_m2 = az.from_netcdf('../../data/idata_m2.nc')

print("Groups in m1:", list(idata_m1.groups()))
print("Groups in m2:", list(idata_m2.groups()))

## WAIC

In [ ]:
waic_m1 = az.waic(idata_m1, pointwise=True)
waic_m2 = az.waic(idata_m2, pointwise=True)

print("WAIC comparison:")
print(f"  m1 (no pooling)  : WAIC={waic_m1.elpd_waic*-2:.1f}  p_WAIC={waic_m1.p_waic:.1f}")
print(f"  m2 (hierarchical): WAIC={waic_m2.elpd_waic*-2:.1f}  p_WAIC={waic_m2.p_waic:.1f}")
print()
print("Note: lower WAIC = better out-of-sample prediction")
print("p_WAIC = effective number of parameters (complexity penalty)")

## LOO (Leave-One-Out)

LOO holds out one observation at a time and scores how well the model predicts it.
Each observation here is one developer-month row.

In [ ]:
loo_m1 = az.loo(idata_m1, pointwise=True)
loo_m2 = az.loo(idata_m2, pointwise=True)

print("LOO comparison:")
print(f"  m1 (no pooling)  : LOO={loo_m1.elpd_loo*-2:.1f}  p_LOO={loo_m1.p_loo:.1f}")
print(f"  m2 (hierarchical): LOO={loo_m2.elpd_loo*-2:.1f}  p_LOO={loo_m2.p_loo:.1f}")
print()
# Pareto k diagnostics
k_m1 = loo_m1.pareto_k.values
k_m2 = loo_m2.pareto_k.values
print("Pareto-k diagnostics (values > 0.7 are problematic):")
print(f"  m1: k > 0.5: {(k_m1 > 0.5).sum()}  k > 0.7: {(k_m1 > 0.7).sum()}")
print(f"  m2: k > 0.5: {(k_m2 > 0.5).sum()}  k > 0.7: {(k_m2 > 0.7).sum()}")

## Compare with az.compare()

ArviZ's `compare()` function ranks models and computes the difference in predictive accuracy with standard error.

In [ ]:
comparison = az.compare({'m1_no_pooling': idata_m1, 'm2_hierarchical': idata_m2},
                        ic='loo', scale='deviance')
print(comparison.to_string())
print()
print("Columns:")
print("  elpd_loo : expected log pointwise predictive density (higher = better)")
print("  p_loo    : effective number of parameters")
print("  d_loo    : difference from best model")
print("  weight   : model weight for stacking ensemble")
print("  se       : standard error of elpd_loo")
print("  dse      : standard error of the difference")

## WAIC / LOO per Observation

Which observations does m2 predict better than m1?
These are the observations where partial pooling helps most.

In [ ]:
# pointwise difference: positive = m2 is better for that observation
loo_diff = loo_m2.loo_i.values - loo_m1.loo_i.values  # (1500,)

df = pd.read_csv('../../data/pr_simulated.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Distribution of per-observation LOO differences
ax = axes[0]
ax.hist(loo_diff, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(0,            color='black', lw=1.5, linestyle='--', label='No difference')
ax.axvline(loo_diff.mean(), color='red', lw=2,
           label=f'Mean={loo_diff.mean():.3f}')
ax.set_xlabel("LOO difference (m2 - m1) per observation")
ax.set_ylabel("Count")
ax.set_title("Per-obs LOO: positive = m2 better")
ax.legend()

# Panel 2: Mean LOO improvement by designation
ax = axes[1]
df['loo_diff'] = loo_diff
DESIGNATIONS = ['aSDE', 'SDE-1', 'SDE-2', 'SDE-3', 'SDE-4']
means = df.groupby('designation')['loo_diff'].mean().reindex(DESIGNATIONS)
colors = ['C2' if v > 0 else 'C3' for v in means.values]
ax.barh(range(len(DESIGNATIONS)), means.values, color=colors, alpha=0.8)
ax.axvline(0, color='black', lw=1)
ax.set_yticks(range(len(DESIGNATIONS)))
ax.set_yticklabels(DESIGNATIONS)
ax.set_xlabel("Mean LOO improvement (m2 - m1)")
ax.set_title("Where does m2 help most?\n(green = m2 better, red = m1 better)")

plt.tight_layout()
plt.show()

## Summary

### What WAIC / LOO tell us

- **Lower WAIC / LOO deviance = better** out-of-sample prediction
- **p_WAIC / p_LOO** = effective number of parameters — m2 will have a higher penalty but should still win if partial pooling helps
- **dse** = standard error of the difference — if `|d_loo| > 2 * dse`, the difference is meaningful

### Expected outcome
m2 should outperform m1 because:
1. The hierarchy regularises developer estimates — less overfitting to individual noise
2. Designation and team structure captures real variation — better predictions for extreme developers
3. Partial pooling is especially helpful for smaller groups (SDE-4 has only 25 developers)

### Pareto-k diagnostics
Values > 0.7 flag observations the model fits poorly (high influence points).
If m1 has many high-k observations, it means no-pooling is overconfident on some developers.

### TODO (fine-tuning)
- Fix m1 identifiability (sum-to-zero constraint) before drawing strong conclusions from WAIC
- Consider Negative Binomial if var/mean > 1.5 in real data